# Audit evidence: Olist delivery and reviews

This notebook first reproduces the submitted code's headline results, then measures the seller multiplicity, review-selection, seller-denominator, and effect-sign checks. The calculations use `src/olist_audit.py`, which calls shared interval and multiple-testing helpers in `src/audit_tools.py`. Only aggregate results are displayed.

In [1]:
import json
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "olist_audit.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from olist_audit import audit_olist

audit = audit_olist()
headline = audit["headline_reproduction"]
print("Submitted-code headline reproduction")
print(json.dumps({key: value for key, value in headline.items()}, indent=2))

Submitted-code headline reproduction
{
  "reviewed_delivered_orders": 95824,
  "platform_late_rate_reviewed": 0.07994865586909333,
  "by_delivery_status": [
    {
      "delivery_status": "Late",
      "order_count": 7661,
      "mean_review_score": 2.565069834225297,
      "median_review_score": 2.0,
      "one_star_share": 0.46208066832006267,
      "five_star_share": 0.22229473959013182
    },
    {
      "delivery_status": "On time",
      "order_count": 88163,
      "mean_review_score": 4.294114310992139,
      "median_review_score": 5.0,
      "one_star_share": 0.0659120039018636,
      "five_star_share": 0.6243548881049874
    }
  ],
  "by_lateness_bucket": [
    {
      "lateness_bucket": "on time / early",
      "order_count": 88163,
      "mean_review_score": 4.294114310992139
    },
    {
      "lateness_bucket": "1-3 days late",
      "order_count": 2636,
      "mean_review_score": 3.7659332321699543
    },
    {
      "lateness_bucket": "4-7 days late",
      "order_count"

## Main score contrast and reported effect direction

The first output is independently calculated from the same filter and review rule as the submitted script. The rank-biserial effect below is oriented so a positive value means on-time scores tend to be higher.

In [2]:
from audit_tools import wilson_interval

checks = audit["checks"]
effect = checks["late_vs_on_time_score"]
print("Late minus on-time mean score:", round(effect["late_minus_on_time_mean"], 3))
print("Customer-clustered 95% CI:", [round(x, 3) for x in effect["clustered_ci_95"]])
print("Correct rank-biserial (positive = on-time higher):", effect["rank_biserial_on_time_higher"])
print("One-star late-minus-on-time difference:", checks["one_star_late_minus_on_time"])
print("Five-star late-minus-on-time difference:", checks["five_star_late_minus_on_time"])
print("Unadjusted seller-flag share Wilson interval:", checks["seller_flags"]["seller_late_rate_ci_95"])

Late minus on-time mean score: -1.729
Customer-clustered 95% CI: [-1.767, -1.691]
Correct rank-biserial (positive = on-time higher): {'estimate': 0.5537367053395655, 'ci_95': [0.5422746783684632, 0.5656361549509292]}
One-star late-minus-on-time difference: {'difference': 0.3961686644181991, 'ci_95': [0.384903858706806, 0.4074655836950909]}
Five-star late-minus-on-time difference: {'difference': -0.4020601485148556, 'ci_95': [-0.41176969921207857, -0.3920838097983521]}
Unadjusted seller-flag share Wilson interval: [0.09713132152543555, 0.14829191373747]


## Seller flags and denominator

The code tests many sellers independently against the same platform rate. The exact binomial p-values are adjusted with Benjamini-Hochberg and Bonferroni through `audit_tools.adjust_pvalues`. The alternative denominator uses all delivered orders with known dates, not only orders with a review.

In [3]:
seller_checks = checks["seller_flags"]
denominators = checks["seller_denominator_comparison"]
print("Seller significance checks:")
print(json.dumps(seller_checks, indent=2))
print("Median seller-rate change, all delivered minus reviewed-only:", denominators["median_late_rate_difference_all_minus_reviewed"])
print("Bootstrap 95% CI:", denominators["median_difference_bootstrap_ci_95"])
print("Seller thresholds:")
print(json.dumps({
    "reviewed_only": denominators["reviewed_order_seller_rates"],
    "all_delivered": denominators["all_delivered_order_seller_rates"],
}, indent=2))

Seller significance checks:
{
  "eligible_sellers_tested": 623,
  "wilson_flags_unadjusted": 75,
  "bh_survivors": 26,
  "bonferroni_survivors": 13,
  "p_value_reference_rate": 0.07994865586909333,
  "seller_late_rate_ci_95": [
    0.09713132152543555,
    0.14829191373747
  ]
}
Median seller-rate change, all delivered minus reviewed-only: 0.0012229210342417918
Bootstrap 95% CI: [-0.004761904761904759, 0.008722844235366073]
Seller thresholds:
{
  "reviewed_only": {
    "seller_count": 2965,
    "eligible_seller_count": 623,
    "platform_late_rate": 0.07994865586909333,
    "unadjusted_wilson_flags": 75,
    "seller_rate_quantiles": {
      "0.1": 0.02030318674613977,
      "0.25": 0.03478523895946763,
      "0.5": 0.06481481481481481,
      "0.75": 0.10236220472440945,
      "0.9": 0.14436090225563922,
      "0.99": 0.25570509648688766
    },
    "share_above_10_percent": 0.2712680577849117,
    "share_above_20_percent": 0.028892455858747994
  },
  "all_delivered": {
    "seller_count

## Review selection and adjustment

Review-score results are necessarily conditional on having a review. Compare observable late rates, freight, and item counts for reviewed and non-reviewed delivered orders. The adjusted linear model is an associational sensitivity check with customer-state fixed effects and customer-clustered uncertainty; product category and geographic distance are not available in the supplied files.

In [4]:
print("Review-selection comparison:")
print(json.dumps(checks["review_selection"], indent=2))
print("Adjusted reviewed-order score model:")
print(json.dumps(checks["adjusted_score_model"], indent=2))
print("Filter log:")
print(json.dumps(checks["filter_log"], indent=2))
print("Multi-seller order count:", checks["multi_seller_orders"])

Review-selection comparison:
{
  "delivered_orders_with_known_dates": 96470,
  "reviewed_orders": 95824,
  "not_reviewed_orders": 646,
  "reviewed_share": 0.993303617704986,
  "reviewed_late_rate": 0.07994865586909333,
  "not_reviewed_late_rate": 0.25541795665634676,
  "late_rate_difference_reviewed_minus_not_reviewed": -0.17546930078725342,
  "difference_ci_95": [
    -0.21051699561126297,
    -0.14330773043499762
  ],
  "mean_total_freight_reviewed": 22.762877149774585,
  "mean_total_freight_not_reviewed": 26.18569659442724,
  "mean_items_reviewed": 1.1412798463850393,
  "mean_items_not_reviewed": 1.2801857585139318,
  "all_delivered_late_rate": 0.08112366538820359
}
Adjusted reviewed-order score model:
{
  "late_coefficient": -1.7128716422291868,
  "ci_95": [
    -1.7508735947662477,
    -1.6748696896921258
  ],
  "n_orders": 95824,
  "covariates": [
    "log total freight",
    "item count",
    "customer state fixed effects"
  ],
  "limitations": [
    "observational association, 